# Ensemble Layer — Reliability, Not a Magic Accuracy Boost

Trains N independently-seeded SNNs on the **identical BCPO-selected features and identical
Bayesian-tuned hyperparameters** -- only random weight initialization differs -- then averages
their predicted probabilities before making a final call.

### What this actually showed, honestly, on real data
| | Accuracy | F1 |
|---|---|---|
| 9 individual models | 94.61% ± 0.67% | 0.958 ± 0.005 |
| Ensemble (averaged) | 94.51% | 0.958 |

**The ensemble does not beat the average individual run.** It does eliminate the risk of the worst
run (93.16% for one unlucky seed) ever being the number you report or deploy, and it's the
methodologically correct way to report a result at all -- a mean over multiple runs, not one
cherry-picked seed. Frame this in your paper as a **reliability contribution**, not a raw accuracy
claim -- the data doesn't support the stronger claim, and overstating it is exactly the kind of
thing a reviewer catches immediately.


In [ ]:
from google.colab import files
import os

uploaded = files.upload()

print("Uploaded files:")
for filename in uploaded.keys():
    print(filename)

# Check that required files exist
TRAIN_CSV = '/content/algae_dataset_train.csv'
TEST_CSV = '/content/algae_dataset_test.csv'

if os.path.exists(TRAIN_CSV) and os.path.exists(TEST_CSV):
    print("\nBoth CSV files uploaded successfully.")
else:
    print("\nPlease upload both algae_dataset_train.csv and algae_dataset_test.csv")

## 1. Config

In [ ]:
TRAIN_CSV = '/content/algae_dataset_train.csv'
TEST_CSV  = '/content/algae_dataset_test.csv'
RANDOM_SEED = 42

CONFIG = dict(hidden_units=8, beta=0.506, lr=0.00907, num_steps=38)   # unchanged Bayesian-tuned values
N_MODELS = 9
TRAIN_EPOCHS = 150

## 2. Install & imports

In [ ]:
!pip install snntorch scikit-learn -q

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import snntorch as snn
from snntorch import surrogate

np.random.seed(RANDOM_SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## 3. Load data, BCPO feature selection (unchanged mechanism)

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)
non_feature_cols = ['longitude', 'latitude', 'cyan_dn', 'label', 'source_date', 'is_synthetic']
feature_columns = [c for c in train_df.columns if c not in non_feature_cols]
assert len(feature_columns) == 31

X_train_raw = train_df[feature_columns].values.astype(np.float64)
y_train = train_df['label'].values.astype(np.int64)
X_test_raw = test_df[feature_columns].values.astype(np.float64)
y_test = test_df['label'].values.astype(np.int64)

scaler = StandardScaler().fit(X_train_raw)
X_train = scaler.transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def binarize(v, rng):
    return (rng.random(len(v)) < sigmoid(v)).astype(int)

_rng_sub = np.random.default_rng(RANDOM_SEED)
sub0 = _rng_sub.choice(np.where(y_train == 0)[0], size=min(1500, (y_train == 0).sum()), replace=False)
sub1 = _rng_sub.choice(np.where(y_train == 1)[0], size=min(1500, (y_train == 1).sum()), replace=False)
fitness_idx = np.concatenate([sub0, sub1])
X_fit, y_fit = X_train[fitness_idx], y_train[fitness_idx]

def fitness_function(mask, X=X_fit, y=y_fit, alpha=0.9, cv=3):
    sel = np.where(mask == 1)[0]
    if len(sel) == 0:
        return 1.0
    clf = KNeighborsClassifier(n_neighbors=3)
    scores = cross_val_score(clf, X[:, sel], y, cv=cv)
    return alpha * (1 - scores.mean()) + (1 - alpha) * (len(sel) / X.shape[1])

class BCPO:
    def __init__(self, n_agents=20, n_iter=40, dim=31, seed=RANDOM_SEED):
        self.n_agents, self.n_iter, self.dim = n_agents, n_iter, dim
        self.rng = np.random.default_rng(seed)
    def optimize(self):
        pos = self.rng.uniform(-1, 1, size=(self.n_agents, self.dim))
        fitness = np.array([fitness_function(binarize(pos[i], self.rng)) for i in range(self.n_agents)])
        bi = int(np.argmin(fitness)); best_pos, best_fit = pos[bi].copy(), fitness[bi]
        for t in range(self.n_iter):
            T = 2 * (1 - t / self.n_iter)
            for i in range(self.n_agents):
                r = self.rng.random(); a, b = self.rng.integers(0, self.n_agents, size=2)
                if r < 0.25: new_pos = pos[i] + self.rng.standard_normal(self.dim) * (pos[a] - pos[b])
                elif r < 0.5: new_pos = pos[i] + self.rng.random(self.dim) * (pos.mean(axis=0) - pos[i])
                elif r < 0.75: new_pos = best_pos + T * self.rng.standard_normal(self.dim) * (best_pos - pos[i])
                else: new_pos = best_pos + self.rng.random(self.dim) * (best_pos - pos[i]) * 0.5
                new_pos = np.clip(new_pos, -3, 3); new_bin = binarize(new_pos, self.rng)
                new_fit = fitness_function(new_bin)
                if new_fit < fitness[i]:
                    pos[i], fitness[i] = new_pos, new_fit
                    if new_fit < best_fit: best_fit, best_pos = new_fit, new_pos.copy()
        mask = binarize(best_pos, self.rng)
        if mask.sum() == 0: mask[np.argmax(best_pos)] = 1
        return mask

mask = BCPO().optimize()
sel_idx = np.where(mask == 1)[0]
selected_features = [feature_columns[i] for i in sel_idx]
print(f'Selected {len(selected_features)}/31 features:', selected_features)

X_train_sel = X_train[:, sel_idx]
X_test_sel = X_test[:, sel_idx]

## 4. SNN architecture (unchanged from your current champion)

In [ ]:
class BaselineSNN(nn.Module):
    def __init__(self, ni, nh, no, beta):
        super().__init__()
        sg = surrogate.fast_sigmoid()
        self.fc1 = nn.Linear(ni, nh); self.lif1 = snn.Leaky(beta=beta, spike_grad=sg)
        self.fc2 = nn.Linear(nh, no); self.lif2 = snn.Leaky(beta=beta, spike_grad=sg)

    def forward(self, x, num_steps):
        m1 = self.lif1.init_leaky(); m2 = self.lif2.init_leaky(); rec = []
        for _ in range(num_steps):
            c1 = self.fc1(x); s1, m1 = self.lif1(c1, m1)
            c2 = self.fc2(s1); s2, m2 = self.lif2(c2, m2)
            rec.append(s2)
        return torch.stack(rec, dim=0)

## 5. Train {N_MODELS} independently-seeded models, evaluate each individually

In [ ]:
Xtr = torch.tensor(X_train_sel, dtype=torch.float32, device=device)
ytr = torch.tensor(y_train, dtype=torch.long, device=device)
Xte = torch.tensor(X_test_sel, dtype=torch.float32, device=device)

individual_results = []
all_test_probs = []

for seed in range(N_MODELS):
    torch.manual_seed(seed)
    model = BaselineSNN(len(sel_idx), CONFIG['hidden_units'], 2, CONFIG['beta']).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(TRAIN_EPOCHS):
        model.train()
        opt.zero_grad()
        loss = loss_fn(model(Xtr, CONFIG['num_steps']).sum(dim=0), ytr)
        loss.backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        spk_counts = model(Xte, CONFIG['num_steps']).sum(dim=0)
        probs = torch.softmax(spk_counts, dim=1).cpu().numpy()
        preds = spk_counts.argmax(dim=1).cpu().numpy()

    acc, f1 = accuracy_score(y_test, preds), f1_score(y_test, preds)
    individual_results.append((acc, f1))
    all_test_probs.append(probs)
    print(f'seed={seed}  acc={acc*100:.2f}%  f1={f1:.4f}')

## 6. Ensemble (average probabilities across all models) and honest comparison

In [ ]:
accs = np.array([r[0] for r in individual_results])
f1s = np.array([r[1] for r in individual_results])

print(f'Individual models: mean acc={accs.mean()*100:.2f}% (std={accs.std()*100:.2f})')
print(f'Individual models: mean f1={f1s.mean():.4f} (std={f1s.std():.4f})')
print(f'Individual models: min acc={accs.min()*100:.2f}%  max acc={accs.max()*100:.2f}%')

ensemble_probs = np.mean(all_test_probs, axis=0)
ensemble_preds = ensemble_probs.argmax(axis=1)
ens_acc = accuracy_score(y_test, ensemble_preds)
ens_f1 = f1_score(y_test, ensemble_preds)

print(f'\nENSEMBLE ({N_MODELS} models): acc={ens_acc*100:.2f}%  f1={ens_f1:.4f}')
print(f'Ensemble vs mean individual: {(ens_acc-accs.mean())*100:+.2f} pts acc')
print(f'Ensemble vs worst individual: {(ens_acc-accs.min())*100:+.2f} pts acc  <- this is the real, honest benefit')
print(f'Ensemble vs best individual: {(ens_acc-accs.max())*100:+.2f} pts acc')

print('\n' + classification_report(y_test, ensemble_preds, target_names=['No Bloom', 'Bloom']))
print(confusion_matrix(y_test, ensemble_preds))

## What to actually claim in your paper
- **Don't claim**: "the ensemble improved accuracy" -- the data shows it landed on the mean, not above it.
- **Do claim**: "ensembling N independently-initialized models eliminates dependence on a single
  random seed, and the reported result (mean/ensemble accuracy) is more statistically defensible
  than any single training run." Cite the specific spread you found (worst run 93.16% vs ensemble
  94.51%) as evidence of why single-run reporting is risky.
- **Also worth stating plainly**: three separate additions were tested for this project (an
  architectural enhancement, a second feature-selection algorithm/hybrid, and this ensemble) and
  none produced a reliable accuracy improvement beyond the tuned baseline -- suggesting the current
  pipeline is close to a genuine performance ceiling for this dataset size and feature set. That is
  itself a legitimate, reportable finding.
